# Non-linear features of the record M (issue #269)

Experiment `experiments/nonlinear_M/nonlinear_M.py`, report in `experiments/nonlinear_M/README.md`. The setting of the open pull requests: Haar U(4) gates on the open brickwork, one fixed circuit per seed, and the **local encoding**: the reference qubit `R` in a Bell pair with the middle system site. The label `Y` is the Z outcome of `R` at the end, which is the initial Z state of the middle site.

Question: a ridge on `vec(M)` reads only `sum_a w_a m_a`. Is the label hidden in non-linear functions of `M` (the absolute value `|M|`, the XOR of neighbouring outcomes, every product `m_a m_b`) that a linear readout cannot see?

The notebook launches the sweep (`RUN = True`) or reads back the latest one, then draws the master figure (accuracy against `p` of every feature map, with the Bayes ceiling of an observer who knows the circuit), the gain over `raw`, the correlation spectrum per order and the space-time map of `|Corr(m_ti, Y)|`.

In [ ]:
import os
from pathlib import Path

# Jupyter starts in the notebook's folder. Move to the repository root (the
# folder that contains pyproject.toml) so conf/, experiments/ and outputs/
# resolve.
root = Path.cwd()
while not (root / "pyproject.toml").is_file() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"cwd: {Path.cwd()}")

## Parameters

Each value can be changed here or through the environment variable of the same name (`tests/test_nonlinear_M_notebook.py` runs a small sweep this way). The sweep goes to `outputs/nonlinear_M/<date_time>/`.

With the defaults (`N = 6, 8`, 10 circuits, 1500 trajectories per `p`, 19 values of `p`, 4 runs in parallel) the sweep takes about 15 minutes. For the cluster sizes (`N = 10, 12`, `PER_P = 8000`, 20 circuits), use `experiments/nonlinear_M/nonlinear_M.sh` directly, then set `RUN = False` and `SWEEP` to that folder.

In [ ]:
N_VALUES = os.environ.get("N_VALUES", "6 8")
SEEDS = os.environ.get("SEEDS", " ".join(str(s) for s in range(42, 52)))
PER_P = os.environ.get("PER_P", "1500")
P_VALUES = os.environ.get("P_VALUES", "")  # empty: the grid of the config
MODES = os.environ.get("MODES", "")  # empty: every mode
MLP = os.environ.get("MLP", "true")
MAX_JOBS = os.environ.get("MAX_JOBS", "4")
OUT_ROOT = Path(os.environ.get("OUT_ROOT", "outputs"))
RUN = os.environ.get("RUN", "0") == "1"
SWEEP = os.environ.get("SWEEP", "")  # a sweep folder to read instead

In [ ]:
import subprocess

from QELM.utils import latest_sweep, new_sweep_dir

if SWEEP:
    sweep_dir = Path(SWEEP)
elif RUN:
    sweep_dir = new_sweep_dir(OUT_ROOT, "nonlinear_M")
    env = {
        **os.environ,
        "OUT_DIR": str(sweep_dir),
        "N_VALUES": N_VALUES,
        "SEEDS": SEEDS,
        "PER_P": PER_P,
        "MLP": MLP,
        "MAX_JOBS": MAX_JOBS,
    }
    if P_VALUES:
        env["P_VALUES"] = P_VALUES
    if MODES:
        env["MODES"] = MODES
    subprocess.run(
        ["bash", "experiments/nonlinear_M/nonlinear_M.sh"], check=True, env=env
    )
else:
    sweep_dir = latest_sweep(OUT_ROOT / "nonlinear_M")
print(f"sweep: {sweep_dir}")

## Figures

`plot_nonlinear_M.py` averages the runs of each `N` over the circuits and writes the figures and `summary.json` into the sweep folder.

In [ ]:
import importlib.util
import json

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

spec = importlib.util.spec_from_file_location(
    "plot_nonlinear_M", "experiments/nonlinear_M/plot_nonlinear_M.py"
)
assert spec is not None and spec.loader is not None
plot_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot_module)
figure_paths = plot_module.main([str(sweep_dir)])
summary = json.loads((sweep_dir / "summary.json").read_text())

fig, axes = plt.subplots(len(figure_paths), 1, figsize=(12, 5 * len(figure_paths)))
for ax, path in zip(axes, figure_paths, strict=True):
    ax.imshow(mpimg.imread(path))
    ax.set_axis_off()
    ax.set_title(path.name)
fig.tight_layout()
plt.show()

## Table

Accuracy averaged over the circuits: the Bayes ceiling, the linear `raw`, the null control `|M|`, the best non-linear map of `M` and the QELM on the final state.

In [ ]:
modes = ["raw", "abs", "raw_abs", "xor_local", "xor_cone", "xor_full"]
for key, entry in summary.items():
    print(f"{key}: {entry['n_circuits']} circuits, T = {int(entry['T'])}")
    print(f"{'p':>6} {'Bayes':>6} {'raw':>6} {'|M|':>6} {'best NL':>8} {'state':>6}")
    for i, p in enumerate(entry["p"]):
        nonlinear = [
            entry[f"accuracy_ridge_{m}"][i]
            for m in modes[2:]
            if f"accuracy_ridge_{m}" in entry
        ]
        best = max(nonlinear) if nonlinear else float("nan")
        state = entry.get("accuracy_ridge_quantum", [float("nan")] * len(entry["p"]))[i]
        raw = entry.get("accuracy_ridge_raw", [float("nan")] * len(entry["p"]))[i]
        absm = entry.get("accuracy_ridge_abs", [float("nan")] * len(entry["p"]))[i]
        print(
            f"{p:6.3f} {entry['accuracy_bayes'][i]:6.3f} {raw:6.3f} {absm:6.3f} "
            f"{best:8.3f} {state:6.3f}"
        )